In [1]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Load the general pretrained BERT model and tokenizer
model_name = "bert-base-uncased"  # General BERT model
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)  # Binary classification
model.eval()

# Load synthetic prompts from file
synthetic_prompt_file = "/content/synthetic_prompts.txt"
with open(synthetic_prompt_file, "r") as file:
    prompts = file.readlines()

# Classify each prompt and count jailbreaking prompts
total_prompts = len(prompts)
jailbreaking_count = 0

# Iterate through prompts and classify
results = []
for prompt in prompts:
    # Tokenize prompt
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True, max_length=512)

    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.nn.functional.softmax(logits, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
        results.append((prompt.strip(), predicted_class))

        # Count jailbreaking prompts
        if predicted_class == 1:
            jailbreaking_count += 1

# Calculate percentage of jailbreaking prompts
jailbreaking_percentage = (jailbreaking_count / total_prompts) * 100

# Print results
print(f"Total Prompts: {total_prompts}")
print(f"Jailbreaking Prompts: {jailbreaking_count}")
print(f"Percentage of Jailbreaking Prompts: {jailbreaking_percentage:.2f}%")

# Save results to a file
output_file = "classified_prompts_general_bert.txt"
with open(output_file, "w") as out_file:
    for prompt, label in results:
        label_text = "Jailbreaking" if label == 1 else "Not Jailbreaking"
        out_file.write(f"{prompt}\t{label_text}\n")

print(f"Classified prompts saved to {output_file}")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total Prompts: 45047
Jailbreaking Prompts: 42873
Percentage of Jailbreaking Prompts: 95.17%
Classified prompts saved to classified_prompts_general_bert.txt
